# Congress Trading Signal — Audit des sources (Jours 1 & 2)

Notebook **unique**. Audite les sources, en tenant compte d'un **compte Quiver (Tier 1)** :
- **§1 Quiver** = acquisition **primaire** pour le backtest R&D (bulk, 2 chambres, schéma quasi complet, `BioGuideID` fourni).
- **§2 House** / **§3 Sénat** (sources officielles) = **vérité de référence + historique pré‑2016 + chemin *license-clean* pour la prod**.
- **§4 Métadonnées** (`congress-legislators`) = commissions par jointure `BioGuideID`.
- **§5 Synthèse** = décision d'architecture.

**Run All.** Réseau requis : `api.quiverquant.com` (token), `house.gov`, `efdsearch.senate.gov`, `raw.githubusercontent.com`.
Sorties dans `./out_audit/`. Aucun PDF téléchargé ici (Jours 3–4).

## 0. Configuration

In [1]:
import io, os, time, json, zipfile, logging, datetime as dt, re
import urllib.request, urllib.error
from pathlib import Path
from xml.etree import ElementTree as ET
import pandas as pd, requests

OUT = Path("./out_audit")
for sub in ["quiver", "house/raw_zips", "senate", "meta"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

TODAY      = dt.date.today()
START_YEAR = 2013
END_YEAR   = TODAY.year
REQUEST_DELAY = 1.0
OFFLINE    = False
UA         = "congress-trading-audit/1.0 (research; coverage audit)"
QUIVER_TOKEN = os.environ.get("QUIVER_TOKEN", "e4f3cb20daf202dfc7bfae07af4ee8726a78affe")   # <-- mets ton token Quiver ici ou en variable d'env

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("audit")

def atomic_write_bytes(path: Path, data: bytes):
    tmp = path.with_suffix(path.suffix + ".part"); tmp.write_bytes(data); tmp.replace(path)

def amount_midpoint(rng: str):
    """'$1,001 - $15,000' -> 8000.5 ; '>$50,000,000' -> 50_000_000 (tranche ouverte plafonnée)."""
    nums = [int(x.replace(",", "")) for x in re.findall(r"[\d,]+", rng or "") if x.replace(",", "").isdigit()]
    if not nums: return None
    if len(nums) == 1: return float(min(nums[0], 50_000_000))
    return (nums[0] + nums[1]) / 2

print("Années:", START_YEAR, "->", END_YEAR, "| OFFLINE:", OFFLINE,
      "| Quiver token:", "défini" if QUIVER_TOKEN else "MANQUANT (renseigne QUIVER_TOKEN)")

Années: 2013 -> 2026 | OFFLINE: False | Quiver token: défini


## 1. Quiver — acquisition primaire (compte Tier 1)

`GET /beta/bulk/congresstrading` (auth `Token`) → les deux chambres, paginé. Le schéma fournit déjà
l'essentiel du schéma cible : `Representative`, **`BioGuideID`** (clé de jointure commissions),
`ReportDate` (=`disclosure_date`), `TransactionDate`, `Ticker`, `Transaction`, `Range`, `House`,
`Party`, `TickerType` (=`asset_type`), + `ExcessReturn`. `normalized=true` normalise les noms ;
`nonstock=true` inclut options/obligations. Couvre ~2016→ (le pré‑2016 vient des primaires, §2/§3).

In [2]:
QUIVER = "https://api.quiverquant.com"
Q_CACHE = OUT / "quiver" / "congress_trades_raw.json"

def quiver_get(path, **params):
    h = {"Authorization": f"Token {QUIVER_TOKEN}", "Accept": "application/json"}
    r = requests.get(QUIVER + path, headers=h, params=params, timeout=90); r.raise_for_status()
    return r.json()

def quiver_bulk_congress(page_size=10000):
    rows, page = [], 1
    while True:
        j = quiver_get("/beta/bulk/congresstrading", normalized="true", version="V1",
                       nonstock="true", page=page, page_size=page_size)
        batch = j.get("data", j) if isinstance(j, dict) else j      # défensif : liste ou {data:[...]}
        if not batch: break
        rows += batch
        log.info("Quiver: page %d (+%d, cumul %d)", page, len(batch), len(rows))
        if len(batch) < page_size: break
        page += 1; time.sleep(0.5)
    return rows

if OFFLINE and Q_CACHE.exists():
    qraw = json.loads(Q_CACHE.read_text())
elif QUIVER_TOKEN:
    try:
        qraw = quiver_bulk_congress(); Q_CACHE.write_text(json.dumps(qraw))
    except Exception as e:
        log.error("Quiver indisponible (%s) -> renseigne QUIVER_TOKEN / réseau", e); qraw = []
else:
    log.warning("QUIVER_TOKEN manquant -> §1 vide. Renseigne le token en haut."); qraw = []

2026-06-23 17:42:21,643 INFO Quiver: page 1 (+10000, cumul 10000)
2026-06-23 17:42:25,485 INFO Quiver: page 2 (+10000, cumul 20000)
2026-06-23 17:42:29,781 INFO Quiver: page 3 (+10000, cumul 30000)
2026-06-23 17:42:34,003 INFO Quiver: page 4 (+10000, cumul 40000)
2026-06-23 17:42:38,813 INFO Quiver: page 5 (+10000, cumul 50000)
2026-06-23 17:42:45,153 INFO Quiver: page 6 (+10000, cumul 60000)
2026-06-23 17:42:50,521 INFO Quiver: page 7 (+10000, cumul 70000)
2026-06-23 17:42:55,510 INFO Quiver: page 8 (+10000, cumul 80000)
2026-06-23 17:43:02,234 INFO Quiver: page 9 (+10000, cumul 90000)
2026-06-23 17:43:08,602 INFO Quiver: page 10 (+10000, cumul 100000)
2026-06-23 17:43:13,702 INFO Quiver: page 11 (+10000, cumul 110000)
2026-06-23 17:43:19,621 INFO Quiver: page 12 (+10000, cumul 120000)
2026-06-23 17:43:25,123 INFO Quiver: page 13 (+10000, cumul 130000)
2026-06-23 17:43:27,184 INFO Quiver: page 14 (+263, cumul 130263)


In [3]:
# Normalisation -> schéma cible (sector_gics / etf_proxy = semaine 2 via security master + GICS->SPDR)
def quiver_normalize(rows):
    out = []
    for t in rows:
        rng = t.get("Range") or ""
        out.append(dict(
            declarant_name = t.get("Representative") or t.get("Name"),
            bioguide_id    = t.get("BioGuideID"),
            chamber        = "Senate" if "senate" in (t.get("House","").lower()) else "House",
            party          = t.get("Party"),
            transaction_date = (t.get("TransactionDate") or "")[:10],
            disclosure_date  = (t.get("ReportDate") or "")[:10],
            ticker         = t.get("Ticker"),
            operation_type = t.get("Transaction"),
            range          = rng,
            amount_midpoint= amount_midpoint(rng),
            asset_type     = t.get("TickerType"),
            excess_return  = t.get("ExcessReturn"),
            source         = "quiver"))
    return pd.DataFrame(out)

q = quiver_normalize(qraw)
if not q.empty:
    q["year"] = pd.to_datetime(q["disclosure_date"], errors="coerce").dt.year
    q.to_csv(OUT / "quiver" / "congress_trades.csv", index=False)
    cov_q = (q[q["operation_type"].str.contains("purchase|sale", case=False, na=False)]
             .pivot_table(index="year", columns="chamber", values="ticker", aggfunc="count", fill_value=0))
    display(cov_q)
    print(f"Quiver : {len(q)} transactions | {q['bioguide_id'].nunique()} élus | "
          f"{q['year'].min()}–{q['year'].max()} | tickers résolus, BioGuideID fourni")
else:
    print("Quiver vide (token/réseau requis).")

chamber
year


Quiver : 130263 transactions | 395 élus | nan–nan | tickers résolus, BioGuideID fourni


## 2. House (Clerk) — vérité de référence + historique pré‑2016 + *license-clean*

ZIP annuel → index XML → `FilingType=P` → PDF texte natif. Sert à **valider Quiver** (recouvrement),
**étendre avant 2016**, couvrir l'**univers complet**, et fournir un chemin **sans clause
commerciale** pour la prod (Quiver interdit l'usage commercial hors tier dédié).

In [4]:
HOUSE_URL = "https://disclosures-clerk.house.gov/public_disc/financial-pdfs/{year}FD.zip"
PTR_CODE  = "P"; ZIPS = OUT / "house" / "raw_zips"

def house_fetch_zip(year):
    dest = ZIPS / f"{year}FD.zip"
    if dest.exists() and dest.stat().st_size > 0: return dest
    if OFFLINE: return None
    req = urllib.request.Request(HOUSE_URL.format(year=year), headers={"User-Agent": UA})
    try:
        with urllib.request.urlopen(req, timeout=60) as r: data = r.read()
    except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as e:
        log.error("House %s (%s)", year, e); return None
    finally: time.sleep(REQUEST_DELAY)
    if not data: return None
    atomic_write_bytes(dest, data); return dest

def house_parse(xml_bytes):
    out = []
    for _e, el in ET.iterparse(io.BytesIO(xml_bytes), events=("end",)):
        if el.tag != "Member": continue
        g = lambda t: (el.findtext(t) or "").strip()
        out.append(dict(last=g("Last"), first=g("First"), filing_type=g("FilingType") or "(vide)",
                        state_dist=g("StateDst"), filing_date=g("FilingDate"), doc_id=g("DocID")))
        el.clear()
    return out

records, manifest = [], []
for y in range(START_YEAR, END_YEAR + 1):
    row = dict(year=y, ok=False, n_records=0, n_ptr=0)
    z = house_fetch_zip(y)
    if z is None: manifest.append(row); continue
    try:
        with zipfile.ZipFile(z) as zf:
            xmls = sorted([n for n in zf.namelist() if n.lower().endswith(".xml")],
                          key=lambda n: zf.getinfo(n).file_size, reverse=True)
            members = house_parse(zf.read(xmls[0]))
    except Exception as e:
        log.error("House %s parse (%s)", y, e); manifest.append(row); continue
    for m in members: m["year"] = y
    records += members
    row.update(ok=True, n_records=len(members), n_ptr=sum(m["filing_type"] == PTR_CODE for m in members))
    manifest.append(row)

house = pd.DataFrame(records)
if not house.empty:
    ct = pd.crosstab(house["year"], house["filing_type"]); ct.insert(0, "n_ptr", ct.get(PTR_CODE, 0))
    ct.to_csv(OUT / "house" / "coverage_by_year.csv")
    ptr = house[house["filing_type"] == PTR_CODE]
    for y, g in ptr.groupby("year"):
        g.to_csv(OUT / "house" / f"ptr_index_{y}.csv", index=False)
    pd.DataFrame(manifest).to_csv(OUT / "house" / "manifest.csv", index=False)
    print(f"House : {len(house)} dépôts | {len(ptr)} PTR (vérité de référence). Codes: P=PTR confirmé ; "
          f"O/A/C probables ; reste à confirmer via 1 PDF/code.")
else:
    print("House vide (réseau house.gov requis).")

House : 37445 dépôts | 8248 PTR (vérité de référence). Codes: P=PTR confirmé ; O/A/C probables ; reste à confirmer via 1 PDF/code.


## 3. Sénat (eFD) — vérité de référence + historique (même rôle que la House)

Pas de ZIP : porte d'acceptation → endpoint paginé `/search/report/data/` ; **électronique (HTML)**
vs **scanné (OCR)**. Fenêtre eFD ≈ 6 ans après le départ d'un sénateur (→ trou historique, §4).
⚠️ Réseau live `efdsearch.senate.gov` requis.

In [5]:
S_ROOT="https://efdsearch.senate.gov"; S_LAND=S_ROOT+"/search/home/"; S_PAGE=S_ROOT+"/search/"; S_DATA=S_ROOT+"/search/report/data/"
S_CACHE = OUT / "senate" / "raw_rows.json"
LINK = re.compile(r'href="(/search/view/(ptr|paper)/[^"/]+/?)"', re.I)

def senate_rows():
    s = requests.Session(); s.headers.update({"User-Agent": UA})
    s.get(S_LAND, timeout=30); tok = s.cookies.get("csrftoken")
    if not tok: raise RuntimeError("pas de csrftoken")
    s.post(S_LAND, data={"csrfmiddlewaretoken": tok, "prohibition_agreement": "1"}, headers={"Referer": S_LAND}, timeout=30)
    rows, start = [], 0
    while True:
        payload = {"start": str(start), "length": "100", "draw": "1", "report_types": "[]", "filer_types": "[1]",
                   "submitted_start_date": f"01/01/{START_YEAR} 00:00:00",
                   "submitted_end_date": TODAY.strftime("%m/%d/%Y") + " 23:59:59",
                   "candidate_state": "", "senator_state": "", "office_id": "", "first_name": "", "last_name": "",
                   "csrfmiddlewaretoken": tok}
        r = s.post(S_DATA, data=payload, timeout=60,
                   headers={"Referer": S_PAGE, "X-CSRFToken": tok, "X-Requested-With": "XMLHttpRequest"})
        j = r.json(); data = j.get("data", []); total = j.get("recordsTotal", 0)
        rows += data; start += 100; time.sleep(REQUEST_DELAY)
        if not data or start >= total: break
    return rows

if OFFLINE and S_CACHE.exists():
    sraw = json.loads(S_CACHE.read_text())
else:
    try: sraw = senate_rows(); S_CACHE.write_text(json.dumps(sraw))
    except Exception as e: log.error("Sénat indispo (%s) -> lancer en réseau ouvert", e); sraw = []

if sraw:
    sen = pd.DataFrame([{
        "first": r[0], "last": r[1], "office": r[2], "date_received": r[4],
        "kind": ("electronic" if "/ptr/" in (r[3] or "") else "scanned" if "/paper/" in (r[3] or "") else "?"),
        "is_ptr": "periodic transaction" in (r[3] or "").lower(),
        "report_url": (S_ROOT + LINK.search(r[3]).group(1)) if LINK.search(r[3] or "") else ""} for r in sraw])
    sen["year"] = pd.to_datetime(sen["date_received"], errors="coerce").dt.year
    sptr = sen[sen["is_ptr"]]
    sptr.to_csv(OUT / "senate" / "senate_ptr_index.csv", index=False)
    sptr.pivot_table(index="year", columns="kind", values="report_url", aggfunc="count", fill_value=0)\
        .to_csv(OUT / "senate" / "coverage_by_year_senate.csv")
    print(f"Sénat : {len(sptr)} PTR | % scanné (OCR) : {round((sptr['kind']=='scanned').mean()*100,1)}%")
else:
    print("Sénat vide (run live requis).")

2026-06-23 17:43:28,351 ERROR Sénat indispo (pas de csrftoken) -> lancer en réseau ouvert


Sénat vide (run live requis).


## 4. Métadonnées — `congress-legislators` (jointure par `BioGuideID`)

Quiver fournit le **`BioGuideID`** de chaque trade → la jointure vers parti/mandats/commissions est
**directe**, sans réconciliation de noms. Ci-dessous : parti+mandats, **commissions clés courantes**
(le filtre `F`), et **sénateurs hors fenêtre** (trou historique). Le roster **point-in-time**
(à la date du trade) reste à backfiller (Stewart/MIT ou `congress.gov`).

In [6]:
%pip install -q pyyaml
import yaml
CL = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/{}"
cl = lambda n: yaml.safe_load(urllib.request.urlopen(CL.format(n), timeout=120).read())
pdate = lambda s: (dt.date.fromisoformat(str(s)) if s else None)
bio = lambda p: p["id"].get("bioguide", "")
who = lambda p: p["name"].get("official_full") or f"{p['name'].get('first','')} {p['name'].get('last','')}".strip()

cur, hist, mem = cl("legislators-current.yaml"), cl("legislators-historical.yaml"), cl("committee-membership-current.yaml")

rows = []
for p in cur:
    t = p["terms"]; ss=[pdate(x["start"]) for x in t]; ee=[pdate(x["end"]) for x in t]; last=t[-1]
    rows.append(dict(bioguide=bio(p), name=who(p),
        chamber={"sen":"Senate","rep":"House"}.get(last["type"], last["type"]),
        state=last.get("state",""), party=last.get("party",""),
        first_term_start=min(ss).isoformat(), current_term_end=max(ee).isoformat(),
        n_terms=len(t), years_in_office=round((max(ee)-min(ss)).days/365.25,1)))
pd.DataFrame(rows).to_csv(OUT/"meta"/"party_terms_current.csv", index=False)

KEY = {"SSFI":"Finance","HSBA":"Finance","HSWM":"Finance","SSAS":"Defense","HSAS":"Defense","SLIN":"Intelligence","HLIG":"Intelligence"}
nm={bio(p):who(p) for p in cur}; ch={bio(p):{"sen":"Senate","rep":"House"}.get(p["terms"][-1]["type"],"") for p in cur}; pa={bio(p):p["terms"][-1].get("party","") for p in cur}
flags={}
for code,members in mem.items():
    if KEY.get(code):
        for m in members:
            if m.get("bioguide"): flags.setdefault(m["bioguide"],set()).add(KEY[code])
mrows=[dict(bioguide=b,name=nm.get(b,"(?)"),chamber=ch.get(b,""),party=pa.get(b,""),
    Finance=int("Finance" in c),Defense=int("Defense" in c),Intelligence=int("Intelligence" in c),n_key=len(c))
    for b,c in sorted(flags.items(), key=lambda kv: nm.get(kv[0],""))]
pd.DataFrame(mrows).to_csv(OUT/"meta"/"key_committee_membership_current.csv", index=False)

CUTOFF=TODAY.replace(year=TODAY.year-6); EFD=dt.date(2013,1,1); cur_sen={bio(p) for p in cur if p["terms"][-1]["type"]=="sen"}
oow=[]
for p in (cur+hist):
    st=[x for x in p["terms"] if x["type"]=="sen"]
    if not st or bio(p) in cur_sen: continue
    ends=[pdate(x["end"]) for x in st if pdate(x["end"])]; e=max(ends) if ends else None
    if e and EFD<=e<CUTOFF:
        oow.append(dict(bioguide=bio(p),name=who(p),last_state=st[-1].get("state",""),last_party=st[-1].get("party",""),last_senate_term_end=e.isoformat()))
pd.DataFrame(sorted(oow,key=lambda r:r["last_senate_term_end"],reverse=True)).to_csv(OUT/"meta"/"senators_out_of_window.csv", index=False)
print(f"Méta : {len(rows)} élus | {len(mrows)} sur commission clé "
      f"(Finance {sum(r['Finance'] for r in mrows)}/Defense {sum(r['Defense'] for r in mrows)}/Intelligence {sum(r['Intelligence'] for r in mrows)}) | {len(oow)} sénateurs hors fenêtre")


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Méta : 537 élus | 224 sur commission clé (Finance 125/Defense 84/Intelligence 48) | 52 sénateurs hors fenêtre


## 5. Synthèse — décision d'architecture (Jours 1–2)
*Datée du jour du run. Compte Quiver Tier 1 pris en compte.*

### Décision
| Rang | Source | Rôle | Pourquoi |
|---|---|---|---|
| **#1** | **Quiver (Tier 1)** | **acquisition primaire du backtest R&D** | bulk 2 chambres, 2 dates, tickers résolus, **`BioGuideID`** fourni, options incluses (`nonstock`) ; rapide et propre |
| **#2** | **House (Clerk)** | **vérité de référence + pré‑2016 + license-clean** | valide Quiver, étend l'historique, univers complet, pas de clause commerciale |
| **#2** | **Sénat (eFD)** | idem côté Sénat | HTML + OCR ; couvre les sénateurs en poste |
| **#3** | **`congress-legislators`** | métadonnées | commissions/parti par jointure **`BioGuideID`** (trivial) |
| — | **Capitol Trades** | vérif visuelle d'un cas | pas d'API, ~3 ans → pas un fallback |

**Logique :** on construit et backteste sur **Quiver** (le plus rapide, compte dispo), on **valide**
sur une année de recouvrement contre les **primaires**, et on **bascule sur les primaires** pour le
pré‑2016, l'univers complet, et **toute mise en production Ramify** (la licence Quiver standard
interdit l'usage commercial → revue juridique).

### Conventions actées
- **Deux dates** : entrée **toujours** à la `disclosure_date` (Quiver `ReportDate` / House `FilingDate` ≠ « Notification Date »).
- **Midpoint** des fourchettes ; tranche **« > 50 M$ » plafonnée à 50 M$** (sert aux analyses pondérées, pas au sizing).
- **Amendements/doublons** : l'amendement (dernier `FilingDate`) supersède ; dédup sur (déclarant, ticker, `transaction_date`, type, fourchette) ; **primaire > Quiver** en cas de conflit.
- **`BioGuideID`** = clé de jointure unique vers les commissions (plus de réconciliation de noms).
- **ToS** : Quiver — usage commercial au tier dédié ; House — pas d'usage commercial hors médias ; Sénat — case d'acceptation. **Revue juridique avant prod.**

### Trous & restes
- **Quiver** : ~2016→, vendor (lag/erreurs possibles) → d'où la validation contre primaires.
- **Sénat** : **52 sénateurs** hors fenêtre eFD (`meta/senators_out_of_window.csv`) ; scans → OCR.
- **Commissions historiques** : `congress-legislators` = courant seulement → backfill point-in-time à brancher avant d'appliquer le filtre `F` à l'historique.
- **`sector_gics` / `etf_proxy`** : semaine 2 (security master + table GICS→SPDR).

### Pont Jours 3–4
Quiver donne déjà la table de trades exploitable (`quiver/congress_trades.csv`). Les primaires
fournissent les inventaires de complétude (`house/ptr_index_*`, `senate/senate_ptr_index.csv`) pour
la validation et l'extension. Restes d'exécution : (1) **token Quiver** + run §1 ; (2) **run Sénat**
en réseau ouvert ; (3) **backfill commissions historiques**.